# Week1_ex2 — Three-Bus Meshed System

Reproduces Example 1.3 from Glover, Overbye & Sarma, *Power System Analysis and Design* (7th ed.). Closing the loop between Bus 2 and Bus 3 is the first case where power can split between parallel paths.

![PowerWorld one-line](Images/Week1_ex2_result_image.png)

In [1]:
import numpy as np
import pandas as pd
import pandapower as pp
import pandapower.topology as top

# 1. Per-unit bases used by the PowerWorld case
S_BASE_MVA = 100.0                      # back-calculated: I = 0.05 pu, I^2 R = 0.00075 pu = 0.075 MW only on 100 MVA
V_BASE_KV = 16.0                        # nominal voltage of every bus
Z_BASE_OHM = V_BASE_KV**2 / S_BASE_MVA  # 2.56 ohm

# 2. Line data read once from PowerWorld Simulator (Run Mode, Model Explorer > Network > Branches Input)
R_PU, X_PU, B_PU = 0.30, 0.60, 0.0
LIM_MVA = 20.0

# 3. Convert per-unit impedance to pandapower inputs (length fixed at 1 km, so ohm/km = ohm)
r_ohm = R_PU * Z_BASE_OHM
x_ohm = X_PU * Z_BASE_OHM
max_i_ka = LIM_MVA / (np.sqrt(3) * V_BASE_KV)   # MVA rating -> current rating at nominal voltage

pd.DataFrame({
    "Quantity": ["Z_base [ohm]", "R [ohm]", "X [ohm]", "I_max [kA]"],
    "Value": [Z_BASE_OHM, r_ohm, x_ohm, max_i_ka],
}).round(4)

,Quantity,Value
0,Z_base [ohm],2.5600
1,R [ohm],0.7680
2,X [ohm],1.5360
3,I_max [kA],0.7217


In [2]:
# 4. Build the network: three identical lines forming a loop, loads at Bus 2 and Bus 3
net = pp.create_empty_network(f_hz=60, sn_mva=S_BASE_MVA)
buses = [pp.create_bus(net, V_BASE_KV, name=f"Bus {i}") for i in (1, 2, 3)]
pp.create_ext_grid(net, buses[0], vm_pu=1.0)
for f, t in [(0, 1), (0, 2), (1, 2)]:
    pp.create_line_from_parameters(net, buses[f], buses[t], length_km=1.0, r_ohm_per_km=r_ohm,
                                   x_ohm_per_km=x_ohm, c_nf_per_km=0.0, max_i_ka=max_i_ka,
                                   name=f"{f + 1}-{t + 1}")
pp.create_load(net, buses[1], p_mw=5.0, q_mvar=0.0)
pp.create_load(net, buses[2], p_mw=10.0, q_mvar=0.0)

# 5. Solve
pp.runpp(net, numba=False)

# 6. Bus results against PowerWorld (Run Mode, Model Explorer > Network > Buses)
pw_bus = pd.DataFrame({"PW V [pu]": [1.00000, 0.97845, 0.97313],
                       "PW angle [deg]": [0.00, -2.34, -2.94]})
bus_cmp = pd.concat([net.bus.name.rename("Bus"), pw_bus,
                     net.res_bus.vm_pu.rename("pp V [pu]"),
                     net.res_bus.va_degree.rename("pp angle [deg]")], axis=1)
bus_cmp["dV [pu]"] = bus_cmp["pp V [pu]"] - bus_cmp["PW V [pu]"]
assert bus_cmp["dV [pu]"].abs().max() < 1e-4
bus_cmp.round(5)

,Bus,PW V [pu],PW angle [deg],pp V [pu],pp angle [deg],dV [pu]
0,Bus 1,1.00000,0.00,1.00000,0.00000,0.00000
1,Bus 2,0.97845,-2.34,0.97845,-2.34312,0.00000
2,Bus 3,0.97313,-2.94,0.97312,-2.94504,-0.00001


In [3]:
# 7. Line flows and losses against PowerWorld (Model Explorer > Network > Branches State)
pw_line = pd.DataFrame({"PW P from [MW]": [6.8, 8.5, 1.7],
                        "PW P loss [MW]": [0.14, 0.22, 0.01],
                        "PW loading [%]": [34.2, 42.8, 8.4]})
line_cmp = pd.concat([net.line.name.rename("Line"), pw_line,
                      net.res_line.p_from_mw.rename("pp P from [MW]"),
                      net.res_line.pl_mw.rename("pp P loss [MW]"),
                      net.res_line.loading_percent.rename("pp loading [%]")], axis=1)
assert (line_cmp["pp P from [MW]"] - line_cmp["PW P from [MW]"]).abs().max() < 0.05
line_cmp.round(3)

,Line,PW P from [MW],PW P loss [MW],PW loading [%],pp P from [MW],pp P loss [MW],pp loading [%]
0,1-2,6.8,0.14,34.2,6.825,0.140,34.160
1,1-3,8.5,0.22,42.8,8.544,0.220,42.770
2,2-3,1.7,0.01,8.4,1.685,0.009,8.611


In [4]:
# 8. Slack output and total loss (PowerWorld Generators table: 15.37 MW)
pd.DataFrame({
    "Quantity": ["Slack P [MW]", "Total loss [MW]"],
    "PowerWorld": [15.37, 15.37 - 15.0],
    "pandapower": [net.res_ext_grid.p_mw.iat[0], net.res_line.pl_mw.sum()],
}).round(3)

,Quantity,PowerWorld,pandapower
0,Slack P [MW],15.37,15.368
1,Total loss [MW],0.37,0.368


In [5]:
# 9. Ybus in per unit (PowerWorld: diagonal 1.33 - j2.67, off-diagonal -0.67 + j1.33)
ybus = net._ppc["internal"]["Ybus"].toarray()
pd.DataFrame(ybus, index=net.bus.name, columns=net.bus.name).map(lambda y: f"{y.real:.4f} {y.imag:+.4f}j")

name,Bus 1,Bus 2,Bus 3
name,,,
Bus 1,1.3333 -2.6667j,-0.6667 +1.3333j,-0.6667 +1.3333j
Bus 2,-0.6667 +1.3333j,1.3333 -2.6667j,-0.6667 +1.3333j
Bus 3,-0.6667 +1.3333j,-0.6667 +1.3333j,1.3333 -2.6667j
